In [11]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from openai import OpenAI
import base64
from io import BytesIO
import re

# Step 1: Configure Docling to extract images
pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

# Step 2: Convert document
print("Converting PDF...")
document = converter.convert("pdfs/CLV.pdf")

# Step 3: Get markdown with image placeholders
markdown_text = document.document.export_to_markdown()
print(f"Converted {len(markdown_text)} characters")

# Step 4: Extract images and get descriptions from GPT-4 Vision
client = OpenAI()
image_descriptions = []

print("\nGenerating image descriptions with GPT-4 Vision...")

if hasattr(document.document, 'pictures'):
    pictures = list(document.document.pictures)
    print(f"Found {len(pictures)} images")
    
    for i, pic in enumerate(pictures):
        try:
            # Get the PIL image from the picture - pass document.document as argument
            pil_image = pic.get_image(document.document)
            
            if pil_image is None:
                print(f"  Image {i+1}: No image data available")
                image_descriptions.append(None)
                continue
            
            # Convert to base64
            buffered = BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            
            # Get description from GPT-4 Vision
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{
                    "role": "user",
                    "content": [
                        {
                            "type": "text", 
                            "text": "Describe this image from a climate policy document. Focus on charts, diagrams, maps, infographics, or visual data. Be specific and concise."
                        },
                        {
                            "type": "image_url", 
                            "image_url": {"url": f"data:image/png;base64,{img_base64}"}
                        }
                    ]
                }],
                max_tokens=300
            )
            
            description = response.choices[0].message.content
            image_descriptions.append(description)
            print(f"  Image {i+1}: {description[:80]}...")
            
        except Exception as e:
            print(f"  Image {i+1}: Error - {e}")
            image_descriptions.append(None)

# Step 5: Replace image placeholders with descriptions in markdown
print("\nEmbedding descriptions into markdown...")
markdown_with_descriptions = markdown_text
image_count = 0

# Replace each <!-- image --> with the description
for i, description in enumerate(image_descriptions):
    if description:
        replacement = f"\n**Image {i+1}:** {description}\n"
        markdown_with_descriptions = markdown_with_descriptions.replace(
            "<!-- image -->", 
            replacement, 
            1  # Replace only the first occurrence
        )
        image_count += 1

# Step 6: Save the result
with open("clv_with_ai_descriptions.md", "w") as f:
    f.write(markdown_with_descriptions)

print(f"\n✅ Complete! Replaced {image_count} images with AI descriptions")
print(f"Saved to: clv_with_ai_descriptions.md")


2026-02-02 16:08:17,515 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-02-02 16:08:17,544 - INFO - Going to convert document batch...
2026-02-02 16:08:17,544 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 8e7b949cc226caef8aab3aadca70e8e7
2026-02-02 16:08:17,544 - INFO - Auto OCR model selected ocrmac.
2026-02-02 16:08:17,545 - INFO - Accelerator device: 'mps'


Converting PDF...


2026-02-02 16:08:18,518 - INFO - Accelerator device: 'mps'
2026-02-02 16:08:19,388 - INFO - Processing document CLV.pdf
2026-02-02 16:08:53,492 - INFO - Finished converting document CLV.pdf in 36.00 sec.


Converted 61953 characters

Generating image descriptions with GPT-4 Vision...
Found 71 images


2026-02-02 16:08:58,234 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

  Image 69: The image features a certification badge for "Leadership in Energy and Environme...


2026-02-02 16:12:45,313 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


  Image 70: The image features a central message, "ALL GROUPS WILL WORK TOGETHER TO CREATE A...


2026-02-02 16:12:51,089 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


  Image 71: The image presents a timeline illustrating the annual process for climate policy...

Embedding descriptions into markdown...

✅ Complete! Replaced 71 images with AI descriptions
Saved to: clv_with_ai_descriptions.md
